In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
print(os.listdir('/content/drive/MyDrive/research datasets'))

['gox_ml_samples.csv', 'pump-and-dump-dataset-master']


In [21]:
import os
import glob
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler
import joblib

print("==========================================================")
print("🔍 LAYER 2 BENCHMARK PIPELINE: INITIALIZATION")
print("==========================================================")

BASE_DIR = '/content/drive/MyDrive/research datasets'

GOX_FILE_PATH = os.path.join(BASE_DIR, 'gox_ml_samples.csv')
SAPIENZA_FILE_PATH = os.path.join(
    BASE_DIR,
    'pump-and-dump-dataset-master/labeled_features/features_5S.csv/features_5S.csv'
)

# Fallback resolution for Sapienza directory partition
if not os.path.exists(SAPIENZA_FILE_PATH):
    fallback_path = os.path.join(BASE_DIR, 'pump-and-dump-dataset-master/features_5S.csv')
    if os.path.isdir(fallback_path):
        internal_csvs = glob.glob(os.path.join(fallback_path, '**', '*.csv'), recursive=True)
        if internal_csvs:
            SAPIENZA_FILE_PATH = internal_csvs[0]
    elif os.path.exists(fallback_path):
        SAPIENZA_FILE_PATH = fallback_path

print(f"📍 Verified Mt.Gox Path:   '{GOX_FILE_PATH}'")
print(f"📍 Verified Sapienza Path: '{SAPIENZA_FILE_PATH}'")

if not os.path.exists(GOX_FILE_PATH) or not os.path.exists(SAPIENZA_FILE_PATH):
    raise FileNotFoundError("❌ One or both source files are missing. Check your Google Drive paths!")

# Load raw datasets
gox_df_raw = pd.read_csv(GOX_FILE_PATH)
pnd_df_raw = pd.read_csv(SAPIENZA_FILE_PATH)

print(f"✅ Loaded Mt.Gox Raw:   {len(gox_df_raw):,} rows")
print(f"✅ Loaded Sapienza Raw: {len(pnd_df_raw):,} rows")

🔍 LAYER 2 BENCHMARK PIPELINE: INITIALIZATION
📍 Verified Mt.Gox Path:   '/content/drive/MyDrive/research datasets/gox_ml_samples.csv'
📍 Verified Sapienza Path: '/content/drive/MyDrive/research datasets/pump-and-dump-dataset-master/labeled_features/features_5S.csv/features_5S.csv'
✅ Loaded Mt.Gox Raw:   5,537,252 rows
✅ Loaded Sapienza Raw: 821,307 rows


In [22]:
print("⚙️ Processing Mt.Gox Wash Trading Benchmark...")

def process_mtgox_pipeline(df_raw):
    df = df_raw.copy().dropna(subset=['wash'])

    # Strict validation of binary target
    if not set(df['wash'].unique()).issubset({0, 1, 0.0, 1.0}):
        raise ValueError("❌ Non-binary values found in Mt.Gox target column!")

    y = df['wash'].astype(int)

    feature_cols = [c for c in ['bitcoins', 'price_deviation', 'buyer_24h_trade_count', 'seller_24h_trade_count'] if c in df.columns]
    X = df[feature_cols].copy()
    for c in X.columns:
        X[c] = pd.to_numeric(X[c], errors='coerce')

    # Domain Feature Engineering
    if 'buyer_24h_trade_count' in X.columns and 'seller_24h_trade_count' in X.columns:
        X['trade_count_ratio'] = (X['buyer_24h_trade_count'] - X['seller_24h_trade_count']) / \
                                 (X['buyer_24h_trade_count'] + X['seller_24h_trade_count'] + 1e-8)
    if 'bitcoins' in X.columns:
        X['log_bitcoins'] = np.log1p(np.maximum(0, X['bitcoins']))
    if 'price_deviation' in X.columns:
        X['abs_price_deviation'] = np.abs(X['price_deviation'])

    return X, y, len(df_raw) - len(df)

X_gox, y_gox, gox_dropped = process_mtgox_pipeline(gox_df_raw)

# Order-of-appearance sequential split (70% / 15% / 15%)
n_gox = len(X_gox)
gox_tr_end = int(n_gox * 0.70)
gox_val_end = int(n_gox * 0.85)

X_gox_tr_raw, y_gox_train = X_gox.iloc[:gox_tr_end].copy(), y_gox.iloc[:gox_tr_end].copy()
X_gox_va_raw, y_gox_val = X_gox.iloc[gox_tr_end:gox_val_end].copy(), y_gox.iloc[gox_tr_end:gox_val_end].copy()
X_gox_te_raw, y_gox_test = X_gox.iloc[gox_val_end:].copy(), y_gox.iloc[gox_val_end:].copy()

# Fit Imputer and RobustScaler STRICTLY on Train
gox_medians = X_gox_tr_raw.median(numeric_only=True)
X_gox_tr_imp = X_gox_tr_raw.fillna(gox_medians)
X_gox_va_imp = X_gox_va_raw.fillna(gox_medians)
X_gox_te_imp = X_gox_te_raw.fillna(gox_medians)

scaler_gox = RobustScaler()
X_gox_train = pd.DataFrame(scaler_gox.fit_transform(X_gox_tr_imp), columns=X_gox.columns, index=X_gox_tr_raw.index)
X_gox_val = pd.DataFrame(scaler_gox.transform(X_gox_va_imp), columns=X_gox.columns, index=X_gox_va_raw.index)
X_gox_test = pd.DataFrame(scaler_gox.transform(X_gox_te_imp), columns=X_gox.columns, index=X_gox_te_raw.index)

gox_scale_pos_weight = (len(y_gox_train) - y_gox_train.sum()) / (y_gox_train.sum() + 1e-8)
print(f"✅ Mt.Gox Processed ({len(X_gox):,} rows) | Train Weight: {gox_scale_pos_weight:.2f}")

⚙️ Processing Mt.Gox Wash Trading Benchmark...
✅ Mt.Gox Processed (5,537,252 rows) | Train Weight: 1.54


In [23]:
print("⚙️ Processing Sapienza Pump & Dump Benchmark...")

def process_sapienza_pipeline(df_raw):
    df = df_raw.copy().dropna(subset=['gt'])

    if not set(df['gt'].unique()).issubset({0, 1, 0.0, 1.0}):
        raise ValueError("❌ Non-binary values found in Sapienza target column!")

    y = df['gt'].astype(int)

    # Contiguous event block IDs to isolate 5S window overlaps
    event_block_ids = (y.diff().fillna(0) != 0).cumsum().values

    feature_cols = [c for c in df.columns if c not in ['gt', 'pump_index', 'date', 'symbol']]
    X = df[feature_cols].copy()
    for c in X.columns:
        X[c] = pd.to_numeric(X[c], errors='coerce')
    X = X.dropna(how='all', axis=1)

    # Domain Feature Engineering
    if 'avg_rush_order' in X.columns and 'avg_volume' in X.columns:
        X['rush_volume_interaction'] = X['avg_rush_order'] * X['avg_volume']
    if 'avg_volume' in X.columns:
        X['log_avg_volume'] = np.log1p(np.maximum(0, X['avg_volume']))
    if 'std_price' in X.columns:
        X['log_std_price'] = np.log1p(np.maximum(0, X['std_price']))

    return X, y, event_block_ids, len(df_raw) - len(df)

X_pnd, y_pnd, pnd_blocks, pnd_dropped = process_sapienza_pipeline(pnd_df_raw)

unique_blocks = np.unique(pnd_blocks)
n_blocks = len(unique_blocks)

tr_end = int(n_blocks * 0.70)
va_end = int(n_blocks * 0.85)

tr_blocks = set(unique_blocks[:tr_end])
va_blocks = set(unique_blocks[tr_end:va_end])
te_blocks = set(unique_blocks[va_end:])

# Verify Zero Block Leakage
assert len(tr_blocks.intersection(va_blocks)) == 0, "❌ Leakage: Train and Val share blocks!"
assert len(tr_blocks.intersection(te_blocks)) == 0, "❌ Leakage: Train and Test share blocks!"

pnd_tr_mask = np.isin(pnd_blocks, list(tr_blocks))
pnd_va_mask = np.isin(pnd_blocks, list(va_blocks))
pnd_te_mask = np.isin(pnd_blocks, list(te_blocks))

X_pnd_tr_raw, y_pnd_train = X_pnd[pnd_tr_mask].copy(), y_pnd[pnd_tr_mask].copy()
X_pnd_va_raw, y_pnd_val = X_pnd[pnd_va_mask].copy(), y_pnd[pnd_va_mask].copy()
X_pnd_te_raw, y_pnd_test = X_pnd[pnd_te_mask].copy(), y_pnd[pnd_te_mask].copy()

# Fit Imputer and RobustScaler STRICTLY on Train
pnd_medians = X_pnd_tr_raw.median(numeric_only=True)
X_pnd_tr_imp = X_pnd_tr_raw.fillna(pnd_medians)
X_pnd_va_imp = X_pnd_va_raw.fillna(pnd_medians)
X_pnd_te_imp = X_pnd_te_raw.fillna(pnd_medians)

scaler_pnd = RobustScaler()
X_pnd_train = pd.DataFrame(scaler_pnd.fit_transform(X_pnd_tr_imp), columns=X_pnd.columns, index=X_pnd_tr_raw.index)
X_pnd_val = pd.DataFrame(scaler_pnd.transform(X_pnd_va_imp), columns=X_pnd.columns, index=X_pnd_va_raw.index)
X_pnd_test = pd.DataFrame(scaler_pnd.transform(X_pnd_te_imp), columns=X_pnd.columns, index=X_pnd_te_raw.index)

pnd_scale_pos_weight = (len(y_pnd_train) - y_pnd_train.sum()) / (y_pnd_train.sum() + 1e-8)
print(f"✅ Sapienza Processed ({len(X_pnd):,} rows) | Train Weight: {pnd_scale_pos_weight:.2f}")

⚙️ Processing Sapienza Pump & Dump Benchmark...
✅ Sapienza Processed (821,307 rows) | Train Weight: 2538.36


In [24]:
print("💾 Exporting Parquet Files and Preprocessing Metadata...")

out_dir = os.path.join(BASE_DIR, 'processed_layer2')
os.makedirs(out_dir, exist_ok=True)

# Export Parquets
X_gox_train.assign(target=y_gox_train.values).to_parquet(os.path.join(out_dir, 'mtgox_train.parquet'))
X_gox_val.assign(target=y_gox_val.values).to_parquet(os.path.join(out_dir, 'mtgox_val.parquet'))
X_gox_test.assign(target=y_gox_test.values).to_parquet(os.path.join(out_dir, 'mtgox_test.parquet'))

X_pnd_train.assign(target=y_pnd_train.values).to_parquet(os.path.join(out_dir, 'sapienza_train.parquet'))
X_pnd_val.assign(target=y_pnd_val.values).to_parquet(os.path.join(out_dir, 'sapienza_val.parquet'))
X_pnd_test.assign(target=y_pnd_test.values).to_parquet(os.path.join(out_dir, 'sapienza_test.parquet'))

# Export Metadata
joblib.dump({'scaler': scaler_gox, 'medians': gox_medians, 'scale_pos_weight': gox_scale_pos_weight}, os.path.join(out_dir, 'mtgox_metadata.joblib'))
joblib.dump({'scaler': scaler_pnd, 'medians': pnd_medians, 'scale_pos_weight': pnd_scale_pos_weight}, os.path.join(out_dir, 'sapienza_metadata.joblib'))

print("🎉 All Artifacts Successfully Saved to '/processed_layer2/'")

💾 Exporting Parquet Files and Preprocessing Metadata...
🎉 All Artifacts Successfully Saved to '/processed_layer2/'


In [25]:
print("==========================================================")
print("📊 LAYER 2 FINAL EXECUTION AUDIT REPORT")
print("==========================================================")

print(f"\n1️⃣ SAPIENZA PUMP & DUMP BENCHMARK:")
print(f"   • Raw Rows: {len(pnd_df_raw):,} | Dropped: {pnd_dropped:,} | Processed: {len(X_pnd):,}")
print(f"   • Train Set: {len(X_pnd_train):,} rows | Normal (0): {(y_pnd_train==0).sum():,} | Pump (1): {(y_pnd_train==1).sum():,} ({((y_pnd_train==1).sum()/len(y_pnd_train))*100:.3f}%)")
print(f"   • Val Set:   {len(X_pnd_val):,} rows | Normal (0): {(y_pnd_val==0).sum():,} | Pump (1): {(y_pnd_val==1).sum():,} ({((y_pnd_val==1).sum()/len(y_pnd_val))*100:.3f}%)")
print(f"   • Test Set:  {len(X_pnd_test):,} rows | Normal (0): {(y_pnd_test==0).sum():,} | Pump (1): {(y_pnd_test==1).sum():,} ({((y_pnd_test==1).sum()/len(y_pnd_test))*100:.3f}%)")
print(f"   • Training scale_pos_weight: {pnd_scale_pos_weight:.2f}")

print(f"\n2️⃣ MT.GOX WASH TRADING BENCHMARK:")
print(f"   • Raw Rows: {len(gox_df_raw):,} | Dropped: {gox_dropped:,} | Processed: {len(X_gox):,}")
print(f"   • Train Set: {len(X_gox_train):,} rows | Normal (0): {(y_gox_train==0).sum():,} | Wash (1): {(y_gox_train==1).sum():,} ({((y_gox_train==1).sum()/len(y_gox_train))*100:.2f}%)")
print(f"   • Val Set:   {len(X_gox_val):,} rows | Normal (0): {(y_gox_val==0).sum():,} | Wash (1): {(y_gox_val==1).sum():,} ({((y_gox_val==1).sum()/len(y_gox_val))*100:.2f}%)")
print(f"   • Test Set:  {len(X_gox_test):,} rows | Normal (0): {(y_gox_test==0).sum():,} | Wash (1): {(y_gox_test==1).sum():,} ({((y_gox_test==1).sum()/len(y_gox_test))*100:.2f}%)")
print(f"   • Training scale_pos_weight: {gox_scale_pos_weight:.2f}")

print("\n==========================================================")

📊 LAYER 2 FINAL EXECUTION AUDIT REPORT

1️⃣ SAPIENZA PUMP & DUMP BENCHMARK:
   • Raw Rows: 821,307 | Dropped: 0 | Processed: 821,307
   • Train Set: 563,738 rows | Normal (0): 563,516 | Pump (1): 222 (0.039%)
   • Val Set:   121,234 rows | Normal (0): 121,187 | Pump (1): 47 (0.039%)
   • Test Set:  136,335 rows | Normal (0): 136,287 | Pump (1): 48 (0.035%)
   • Training scale_pos_weight: 2538.36

2️⃣ MT.GOX WASH TRADING BENCHMARK:
   • Raw Rows: 5,537,252 | Dropped: 0 | Processed: 5,537,252
   • Train Set: 3,876,076 rows | Normal (0): 2,350,869 | Wash (1): 1,525,207 (39.35%)
   • Val Set:   830,588 rows | Normal (0): 626,612 | Wash (1): 203,976 (24.56%)
   • Test Set:  830,588 rows | Normal (0): 657,125 | Wash (1): 173,463 (20.88%)
   • Training scale_pos_weight: 1.54



In [30]:
"""TradeSense Component 3 / Layer 2: corrected benchmark partitioning.

Sapienza: chronological, pump_index-group-disjoint split, 1-hour embargo.
Mt.Gox: index-order exploratory split ONLY; chronology/entity independence unverified.
Outputs are written to a NEW folder and never overwrite existing files.
Run in Google Colab after mounting Google Drive.
"""
from pathlib import Path
import json
import shutil
import uuid
import numpy as np
import pandas as pd
import joblib
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

BASE = Path('/content/drive/MyDrive/research datasets')
SAP_FILE = BASE / 'pump-and-dump-dataset-master/labeled_features/features_5S.csv/features_5S.csv'
GOX_FILE = BASE / 'gox_ml_samples.csv'  # Edit if your file lives elsewhere
OUTPUT = BASE / 'processed_layer2_final_v3'
EMBARGO = pd.Timedelta(hours=1)  # Assumption, not a verified lookback bound
RUN_SAPIENZA = True
RUN_MTGOX = True

SAP_BASE = ['std_rush_order', 'avg_rush_order', 'std_trades', 'std_volume',
            'avg_volume', 'std_price', 'avg_price', 'avg_price_max',
            'hour_sin', 'hour_cos', 'minute_sin', 'minute_cos']
GOX_BASE = ['bitcoins', 'price_deviation', 'buyer_24h_trade_count', 'seller_24h_trade_count']


def features(df, dataset):
    cols = SAP_BASE if dataset == 'sapienza' else GOX_BASE
    missing = set(cols) - set(df.columns)
    if missing:
        raise ValueError(f'{dataset}: missing predictor columns: {missing}')
    x = df[cols].apply(pd.to_numeric, errors='coerce').copy()
    if dataset == 'sapienza':
        x['rush_volume_interaction'] = x['avg_rush_order'] * x['avg_volume']
        x['log_avg_volume'] = np.log1p(x['avg_volume'].clip(lower=0))
        x['log_std_price'] = np.log1p(x['std_price'].clip(lower=0))
        assert len(x.columns) == 15
    else:
        x['trade_count_ratio'] = x['buyer_24h_trade_count'] / (x['seller_24h_trade_count'] + 1)
        x['log_bitcoins'] = np.log1p(x['bitcoins'].clip(lower=0))
        x['abs_price_deviation'] = x['price_deviation'].abs()
        assert len(x.columns) == 7
    return x.replace([np.inf, -np.inf], np.nan)


def report_parts(parts, target, name):
    print(f'\n===== {name.upper()} FINAL SPLIT =====')
    for part, frame in parts.items():
        counts = frame[target].value_counts(dropna=False).to_dict()
        print(f'{part:5s}: {len(frame):>10,} rows; labels={counts}')
        if len(frame) == 0 or not frame[target].isin([0, 1]).all() or frame[target].nunique() != 2:
            raise ValueError(f'{name}/{part}: both valid classes are required')


def sapienza_split():
    df = pd.read_csv(SAP_FILE)
    required = {'date', 'pump_index', 'gt'} | set(SAP_BASE)
    if required - set(df.columns):
        raise ValueError(f'Sapienza missing columns: {required - set(df.columns)}')
    df['date'] = pd.to_datetime(df['date'], utc=True, errors='coerce')
    df['pump_index'] = pd.to_numeric(df['pump_index'], errors='coerce')
    if df['date'].isna().any() or not df['gt'].isin([0, 1]).all():
        raise ValueError('Sapienza invalid date or target')

    # FIX: Replaced df.gt.eq(1) with df['gt'] == 1
    if (df.loc[df['gt'] == 1, 'pump_index'].isna() |
        df.loc[df['gt'] == 1, 'pump_index'].lt(0)).any():
        raise ValueError('Some positive observations have no nonnegative pump_index')

    df = df.sort_values('date', kind='mergesort').reset_index(drop=True)
    event = df['pump_index'].ge(0)
    groups = (df.loc[event].groupby('pump_index')['date']
              .agg(start='min', end='max', n='size')
              .sort_values(['start', 'end']))
    if len(groups) < 20:
        raise ValueError('Too few event groups for splitting')

    cut1 = groups['start'].iloc[int(len(groups) * .70)]
    cut2 = groups['start'].iloc[int(len(groups) * .85)]
    if cut1 >= cut2 or cut2 - cut1 <= 2 * EMBARGO:
        raise ValueError('Invalid/too-close temporal cutoffs')
    lo1, hi1 = cut1 - EMBARGO, cut1 + EMBARGO
    lo2, hi2 = cut2 - EMBARGO, cut2 + EMBARGO

    train_ids = set(groups.index[groups['end'] < lo1])
    val_ids = set(groups.index[(groups['start'] >= hi1) & (groups['end'] < lo2)])
    test_ids = set(groups.index[groups['start'] >= hi2])
    assert not (train_ids & val_ids or train_ids & test_ids or val_ids & test_ids)

    background = ~event
    parts = {
        'train': df.loc[(event & df.pump_index.isin(train_ids)) |
                        (background & df.date.lt(lo1))].copy(),
        'val': df.loc[(event & df.pump_index.isin(val_ids)) |
                      (background & df.date.ge(hi1) & df.date.lt(lo2))].copy(),
        'test': df.loc[(event & df.pump_index.isin(test_ids)) |
                       (background & df.date.ge(hi2))].copy(),
    }
    report_parts(parts, 'gt', 'sapienza')
    for label, part in parts.items():
        if not part.date.is_monotonic_increasing:
            raise AssertionError(f'{label} is not time-sorted')
        # FIX: Replaced part.gt.eq(1) with part['gt'] == 1
        print(label, 'positive pump_index groups:', part.loc[part['gt'] == 1, 'pump_index'].nunique(),
              'date range:', part.date.min(), '->', part.date.max())
    if not (parts['train'].date.max() < parts['val'].date.min() < parts['test'].date.min()):
        raise AssertionError('Temporal partition ranges overlap')
    kept = sum(len(p) for p in parts.values())
    print('Cutoffs:', cut1, cut2, '| embargo:', EMBARGO)
    print('Rows excluded (embargo + boundary-spanning event groups):', len(df) - kept)
    print('Event groups excluded:', len(groups) - len(train_ids | val_ids | test_ids))
    return parts, [
        'Sapienza is sorted by parsed date; naive timestamps interpreted as UTC (source timezone unverified).',
        'pump_index>=0 treated as an event-group ID; original documentation semantics not independently verified.',
        'Each retained pump_index group belongs to only one partition; spanning groups are excluded entirely.',
        'One-hour embargo is heuristic, not proven to eliminate all feature-history dependence.',
        'Positive rows are not interchangeable with independent event counts.',
    ]


def mtgox_split():
    df = pd.read_csv(GOX_FILE, usecols=lambda c: c in GOX_BASE + ['wash'])
    if set(GOX_BASE + ['wash']) - set(df.columns):
        raise ValueError('Mt.Gox source is missing required columns')
    n = len(df)
    a, b = int(n * .70), int(n * .85)
    parts = {'train': df.iloc[:a].copy(), 'val': df.iloc[a:b].copy(), 'test': df.iloc[b:].copy()}
    report_parts(parts, 'wash', 'mtgox')
    return parts, [
        'Sequential SOURCE ROW INDEX split only; NOT a verified chronological or group-independent split.',
        'Source contains no timestamp, transaction ID, or entity ID to verify entity leakage.',
        'Historical 24h aggregates may share underlying observations across partitions.',
        'Use results only as exploratory within-dataset benchmarks, not leakage-free prospective performance.',
    ]


def export_dataset(parts, dataset, target, limitations, stage):
    x = {k: features(v, dataset) for k, v in parts.items()}
    columns = x['train'].columns.tolist()
    if any(v.columns.tolist() != columns for v in x.values()):
        raise ValueError('Feature schema differs across splits')
    imputer = SimpleImputer(strategy='median', keep_empty_features=True)
    scaler = RobustScaler()
    train_imputed = imputer.fit_transform(x['train'])
    scaler.fit(train_imputed)
    y = parts['train'][target].astype(int)
    weight = float((y == 0).sum() / (y == 1).sum())
    for split, frame in parts.items():
        arr = scaler.transform(imputer.transform(x[split]))
        if not np.isfinite(arr).all():
            raise ValueError(f'{dataset}/{split}: non-finite predictors')
        out = pd.DataFrame(arr, columns=columns, dtype='float32')
        out[target] = frame[target].astype('int8').to_numpy()
        out.to_parquet(stage / f'{dataset}_{split}.parquet', index=False)
        print(f'Saved {dataset}_{split}.parquet:', out.shape)
        del arr, out
    joblib.dump({'features': columns, 'imputer': imputer, 'scaler': scaler,
                 'scale_pos_weight': weight, 'limitations': limitations},
                stage / f'{dataset}_metadata.joblib')
    print(dataset, 'training scale_pos_weight:', round(weight, 4))


def main():
    if OUTPUT.exists():
        raise FileExistsError(f'Refusing to overwrite existing output: {OUTPUT}')
    if RUN_SAPIENZA and not SAP_FILE.is_file():
        raise FileNotFoundError(SAP_FILE)
    if RUN_MTGOX and not GOX_FILE.is_file():
        raise FileNotFoundError(GOX_FILE)
    stage = BASE / ('.layer2_stage_' + uuid.uuid4().hex)
    stage.mkdir(parents=True, exist_ok=False)
    try:
        if RUN_SAPIENZA:
            parts, notes = sapienza_split()
            export_dataset(parts, 'sapienza', 'gt', notes, stage)
            del parts
        if RUN_MTGOX:
            parts, notes = mtgox_split()
            export_dataset(parts, 'mtgox', 'wash', notes, stage)
            del parts
        stage.rename(OUTPUT)
        print('\nSUCCESS: Final exports saved to:', OUTPUT)
        print('WARNING: Mt.Gox temporal/entity leakage remains UNVERIFIABLE with this source.')
    except Exception:
        shutil.rmtree(stage, ignore_errors=True)
        raise


if __name__ == '__main__':
    main()


===== SAPIENZA FINAL SPLIT =====
train:    600,827 rows; labels={0: 600607, 1: 220}
val  :    121,255 rows; labels={0: 121211, 1: 44}
test :     89,334 rows; labels={0: 89286, 1: 48}
train positive pump_index groups: 220 date range: 2018-01-02 18:14:40+00:00 -> 2020-03-22 16:35:05+00:00
val positive pump_index groups: 44 date range: 2020-03-25 15:59:20+00:00 -> 2020-09-09 16:00:00+00:00
test positive pump_index groups: 48 date range: 2020-09-15 16:23:55+00:00 -> 2021-01-19 16:58:25+00:00
Cutoffs: 2020-03-24 15:57:25+00:00 2020-09-09 18:01:40+00:00 | embargo: 0 days 01:00:00
Rows excluded (embargo + boundary-spanning event groups): 9891
Event groups excluded: 5
Saved sapienza_train.parquet: (600827, 16)
Saved sapienza_val.parquet: (121255, 16)
Saved sapienza_test.parquet: (89334, 16)
sapienza training scale_pos_weight: 2730.0318

===== MTGOX FINAL SPLIT =====
train:  3,876,076 rows; labels={False: 2350869, True: 1525207}
val  :    830,588 rows; labels={False: 626612, True: 203976}
test